In [1]:
!pip install transformers datasets peft accelerate bitsandbytes trl



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from huggingface_hub import login

# login("your_huggingface_token")


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # LLaMA-style, lightweight and open

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_8bit=True,
    device_map="auto"
)


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


In [4]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer
import re
from sklearn.model_selection import train_test_split

In [5]:
# 1. Load datasets using pandas
ncert_df = pd.read_json('datasets/ncert_finetune.json')
human_df = pd.read_json('datasets/human_dataset.json')

In [6]:
# 2. Normalize column names (strip and lower)
ncert_df.columns = ncert_df.columns.str.strip().str.lower()
human_df.columns = human_df.columns.str.strip().str.lower()

In [7]:
# 3. Check if expected columns exist
required_cols = {'prompt', 'response'}
if not required_cols.issubset(ncert_df.columns):
    raise ValueError(f"Missing columns in NCERT dataset: {required_cols - set(ncert_df.columns)}")
if not required_cols.issubset(human_df.columns):
    raise ValueError(f"Missing columns in human dataset: {required_cols - set(human_df.columns)}")

In [8]:
# 4. Combine and shuffle
combined_df = pd.concat([ncert_df[['prompt', 'response']], human_df[['prompt', 'response']]], ignore_index=True)
combined_df = combined_df.sample(frac=1).reset_index(drop=True)

In [9]:
# 5. Train-test split
train_df, test_df = train_test_split(combined_df, test_size=0.1, random_state=42)

In [10]:
# 6. Preprocessing
def clean_text(text):
    text = text.lower().strip()
    return re.sub(r"\s+", " ", text)

for col in ['prompt', 'response']:
    train_df[col] = train_df[col].apply(clean_text)
    test_df[col] = test_df[col].apply(clean_text)

In [11]:
# 8. Format and tokenize
def format_and_tokenize(df):
    formatted = [f"### Prompt:\n{p}\n\n### Response:\n{r}" for p, r in zip(df['prompt'], df['response'])]
    return tokenizer(formatted, truncation=True, padding="max_length", max_length=512)

train_encodings = format_and_tokenize(train_df)
test_encodings = format_and_tokenize(test_df)

In [12]:

# 9. Convert to HF Datasets
train_dataset = Dataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask']
})
test_dataset = Dataset.from_dict({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask']
})


In [13]:
from peft import LoraConfig, get_peft_model, TaskType

# 2. Set up LoRA configuration
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    task_type=TaskType.CAUSAL_LM,
    lora_dropout=0.05,
    bias="none"
)
# 3. Get the LoRA model
model = get_peft_model(model, peft_config)

In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./lora_chatbot",
    per_device_train_batch_size=4,
    num_train_epochs=4,
    learning_rate=2e-4,
    fp16=True,
    save_strategy="epoch",
    logging_steps=10,
    report_to="none",  # Disable wandb
      # You can add this if you want evaluation during training
)


In [15]:
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

# 7. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator
)




No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [16]:
# trainer.train()

In [17]:
model.save_pretrained("./lora_mentor_modelv2")
tokenizer.save_pretrained("./lora_mentor_model_tokenizerv2")


('./lora_mentor_model_tokenizerv2/tokenizer_config.json',
 './lora_mentor_model_tokenizerv2/special_tokens_map.json',
 './lora_mentor_model_tokenizerv2/tokenizer.json')

In [18]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from tqdm import tqdm

In [19]:
# 1. Load saved model and tokenizer
model_path = "./lora_mentor_modelv2"
tokenizer_path = "./lora_mentor_model_tokenizerv2"

model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

model.eval()  # Set model to evaluation mode

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=2048, out_features=8, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=8, out_features=2048, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=256, bia

In [20]:
# 2. Generate responses from the test set
def generate_response(prompt, max_new_tokens=100):
    input_text = f"### Prompt:\n{prompt}\n\n### Response:\n"
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Greedy decoding (for determinism)
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the generated response part
    return response.split("### Response:\n")[-1].strip()


In [23]:
# 3. Test on a few examples from the test dataset
print("\n=== Test Results ===\n")
for i in range(5):  # Change to 10 or more if you want
    prompt = test_df.iloc[i]["prompt"]
    actual_response = test_df.iloc[i]["response"]
    predicted_response = generate_response(prompt)

    print(f"Prompt:\n{prompt}\n")
    print(f"Actual Response:\n{actual_response}\n")
    print(f"Predicted Response:\n{predicted_response}\n")
    print("="*50 + "\n")


=== Test Results ===



Prompt:
given a reference text about ihor lapin, tell me what he did before the war.

Actual Response:
before the war, ihor lapin was a lawyer and a member of the bar qualification-disciplinary commission.

Predicted Response:
before the war, lapin was a successful businessman in his native ukraine. He owned a successful construction company and was a member of the national republican party.


Prompt:
write a program to count the sum of first 100 numbers in python

Actual Response:
here is a simple program that counts the sum of the first 100 numbers in python: ``` # initialize the sum to 0 sum = 0 # loop through the first 100 numbers and add them to the sum for i in range(1, 101): sum += i # print the sum print(sum) ``` this program uses a for loop to iterate over the first 100 numbers (1-100), adding each number to the `sum` variable as it goes. at the end, the program prints the final value of `sum`, which should be the sum of the first 100 numbers.

Predicted Response:
```python
# 

In [24]:
metrics = trainer.evaluate(eval_dataset=test_dataset)
print("Eval Loss:", metrics["eval_loss"])
print("Perplexity:", torch.exp(torch.tensor(metrics["eval_loss"])).item())

Eval Loss: 1.9354983568191528
Perplexity: 6.92749547958374


In [22]:
print(test_df.columns)


Index(['prompt', 'response'], dtype='object')


In [1]:
import zipfile
import os

def zip_folders(folders, output_zip):
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for folder in folders:
            for root, _, files in os.walk(folder):
                for file in files:
                    file_path = os.path.join(root, file)
                    # Add file with a relative path
                    arcname = os.path.relpath(file_path, start=os.path.dirname(folder))
                    zipf.write(file_path, arcname)

model_path = "./lora_mentor_modelv2"
tokenizer_path = "./lora_mentor_model_tokenizerv2"
output_zip = "lora_mentor_model_bundle.zip"

zip_folders([model_path, tokenizer_path], output_zip)

print(f"Zipped to: {output_zip}")


Zipped to: lora_mentor_model_bundle.zip
